# EADA Data Transformation
## Equity in Athletics Disclosure Act

Joe Martin - joseph.martin@unc.edu

Last Updated 10/1/2026

## Introduction

Under the Equity in Athletics Disclosure Act (EADA), the US Department of Education collects data related to all athletic programs at postsecondary institutions that receive Title IV funding. Each year, EADA release two data file called `schools` and `instLevel`. Each row in `schools` consists of one sport at an institution. Each row in `instLevel` contains one row per institution and aggregates certain data values together (for example, this shows the total amount of student aid paid to all men's sports).

Link to new data files: `https://ope.ed.gov/athletics/#/datafile/list`

### Variables of Interest

|   Variable   |  Description  | Data File | OneDrive Folder |
|------------|--------|----------------|------|
| `UNITID` | Unique identifier | schools | |
| `classification_name` | Athletic Organization and Division | schools | |
| `Sports` | Name of sport | schools | |
| `TOTAL_EXPENSE_MEN` | Total expenses for men's team | schools | |
| `TOTAL_EXPENSE_WOMEN` | Total expenses for women's team | schools | |
| `REVENUE_MENALL` | Revenue generated by men's team | schools | |
| `REVENUE_WOMENALL` | Revenue generated by women's team | schools | |
| `PARTIC_MEN` | Count of student-athletes | schools | |
| `PARTIC_WOMEN` | Count of student-athletes | schools | |
| `OPEXPPERTEAM_MEN` | Operating expenses for team | schools | |
| `OPEXPPERTEAM_WOMEN` | Operating expenses for team | schools | |
| `EFMaleCount` | Fall Enrollment Headcount | schools | |
| `EFFemaleCount` | Fall Enrollment Headcount | schools | |
| `UNITID` | Unique Identifier | instLevel | |
| `STUDENTAID_MEN` | Total Value of Student Aid Awarded to Athletes | instLevel | |
| `STUDENTAID_WOMEN` | Total Value of Student Aid Awarded to Athletes | instLevel | |
| `RECRUITEXP_MEN` | Total Recruiting Expenses | instLevel | |
| `RECRUITEXP_WOMEN` | Total Recruiting Expenses | instLevel| |
| `IL_PARTIC_MEN` | Count of student-athletes | instLevel | |
| `IL_PARTIC_WOMEN` | Count of student-athletes | instLevel| |

# Setup
## Libraries and File Paths

In [41]:
# Import libraries and set working directory

import pandas as pd
from os import listdir
from os.path import isfile, join
import numpy as np

from pyprojroot import here 

wd = here('data/raw-data/EADA/schools/')
pub = str(here('data/processed-data'))

# Functions

In [2]:
# I turned the for loop into a function. This should make it easier to create dataframes for each section of files

def create_df(path,files,var_list): # function takes working directory location (path), list of files, and list of variables
    data = pd.DataFrame(columns = var_list)
    for i in range(len(files)):
        try:
            temp = pd.read_excel(path+'/'+files[i]) # open each file and create a temp dataframe
        except: 
            temp = pd.read_csv(path+'/'+files[i], encoding='cp1252')
        temp = temp[var_list] # keep only the columns identified in the variable list
        
        temp['file'] = files[i].removesuffix('.csv').upper().removesuffix('_RV') # each row will be tied back to its original file with the file name
        
        data = pd.concat([data,temp]) # append data to the final dataframe
        
    data = data.reset_index().drop(columns='index') # reset the index 
       
    return data

In [3]:
def get_academic_year(data):
    """
    Adds an 'Academic Year' column based on the year in data['file'].

    Examples:
        SFA2021.csv  -> 2020-21
        HD2023.csv   -> 2022-23
        SFA2223.csv  -> 2022-23
        C202324.csv  -> 2023-24
    """

    result = data.copy()

    year_token = (
        result["file"]
        .astype("string")
        .str.extract(r"(?<!\d)(\d{4})(?!\d)", expand=False)
    )

    result["Academic Year"] = pd.Series(
        pd.NA,
        index=result.index,
        dtype="string"
    )

    # Four-digit calendar year, such as 2021
    calendar_mask = year_token.str.match(r"^20\d{2}$", na=False)

    calendar_year = pd.to_numeric(
        year_token.where(calendar_mask),
        errors="coerce"
    )

    calendar_start = calendar_year - 1

    result.loc[calendar_mask, "Academic Year"] = (
        calendar_start.loc[calendar_mask].astype("Int64").astype("string")
        + "-"
        + calendar_year.loc[calendar_mask]
            .astype("Int64")
            .astype("string")
            .str[-2:]
    )

    # Compact academic year, such as 2223 or 2324
    academic_mask = year_token.notna() & ~calendar_mask

    academic_start = (
        "20" + year_token.loc[academic_mask].str[:2]
    )

    academic_end = year_token.loc[academic_mask].str[2:]

    result.loc[academic_mask, "Academic Year"] = (
        academic_start + "-" + academic_end
    )

    return result

In [25]:
path = str(wd)
files = [f for f in listdir(path) if isfile(join(path, f))] #identify the file names that need to be read

In [26]:
# list of variables needed from these files
var_list = [    
    'unitid'
    ,'classification_name' 	 	
    ,'Sports'
    ,'EXP_MEN'
    ,'EXP_WOMEN'
    ,'REVENUE_MENALL'
    ,'REVENUE_WOMENALL'
    ,'PARTIC_MEN'
    ,'PARTIC_WOMEN' 
    ,'OPEXPPERTEAM_MEN'
    ,'OPEXPPERTEAM_WOMEN'
    ,'EFMaleCount'
    ,'EFFemaleCount'    
]

# setting up new columns to make it easier to create a long dataframe. In a later step, split label on underscore
rename = { 'unitid':'UNITID'
    ,'Sports':'Sport'
    ,'EXP_MEN': 'Expenses_Men'
    ,'EXP_WOMEN': 'Expenses_Women'
    ,'REVENUE_MENALL' : 'Revenue_Men'
    ,'REVENUE_WOMENALL' : 'Revenue_Women'
    ,'PARTIC_MEN': 'Participants_Men'
    ,'PARTIC_WOMEN' : 'Participants_Women'
    ,'OPEXPPERTEAM_MEN' : 'Operational Expenses_Men'
    ,'OPEXPPERTEAM_WOMEN' : 'Operational Expenses_Women'
    ,'EFMaleCount' : 'Student Headcount_Men'
    ,'EFFemaleCount': 'Student Headcount_Women'        
}

# Create empty dataframe with necessary variables
data = pd.DataFrame(columns = var_list)

In [27]:
data = create_df(path, files,var_list)

C:\Users\jmar18\AppData\Local\Temp\ipykernel_79184\1857122136.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data = pd.concat([data,temp]) # append data to the final dataframe


In [28]:
df = data.rename(columns=rename) # rename the columns to be user-friendly. This is helpful when developing dashboards
df['Academic Year'] = df['file'].str.split('_',expand=True)[1].str.split('.',expand=True)[0] # get academic year from file name

In [29]:
df = df.melt(id_vars=['UNITID','Academic Year','classification_name','Sport'], value_vars=df.iloc[0:,3:11])

In [30]:
df['var'] = df['variable'].str.split('_',expand=True)[0]
df['gender'] = df['variable'].str.split('_',expand=True)[1]
df['Fall Term'] = df['Academic Year'].str.split('-',expand=True)[0] # creating a fall term column in case I want to join on fall enrollment data later

In [31]:
df = df.drop(columns='variable')

In [32]:
# move values back into their own columns. This is important for the rank calculation in the Tableau dashboard
df = df.pivot(index=['UNITID','Fall Term','Academic Year','classification_name','Sport','gender'],columns='var',values='value').reset_index()

In [33]:
df['Revenue'] = df['Revenue'].astype(float)

In [34]:
df = df.replace(0,np.nan)

C:\Users\jmar18\AppData\Local\Temp\ipykernel_79184\3706637896.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(0,np.nan)


In [35]:
df

var,UNITID,Fall Term,Academic Year,classification_name,Sport,gender,Expenses,Operational Expenses,Participants,Revenue
0,100654,2020,2020-21,NCAA Division I-FCS,Baseball,Men,253252.0,125769.0,43.0,279387.0
1,100654,2020,2020-21,NCAA Division I-FCS,Baseball,Women,NaN,NaN,NaN,NaN
2,100654,2020,2020-21,NCAA Division I-FCS,Basketball,Men,1296985.0,409876.0,15.0,1107100.0
3,100654,2020,2020-21,NCAA Division I-FCS,Basketball,Women,1071428.0,375175.0,16.0,994674.0
4,100654,2020,2020-21,NCAA Division I-FCS,Bowling,Men,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
171825,800001,2024,2024-25,NCAA Division II without football,Track and Field (Outdoor),Women,263235.0,47184.0,27.0,178347.0
171826,800001,2024,2024-25,NCAA Division II without football,Volleyball,Men,NaN,NaN,NaN,NaN
171827,800001,2024,2024-25,NCAA Division II without football,Volleyball,Women,406078.0,171905.0,18.0,436903.0
171828,800001,2024,2024-25,NCAA Division II without football,Wrestling,Men,266243.0,120518.0,30.0,292754.0


In [42]:
df.to_csv(pub + '/eada_schools_sports.csv')